# 06 - Text Classification con DistilBERT + Optuna
### Adaptado para generar output compatible con el Ensemble (mismo formato que LightGBM tabular y Resnet50 de imagenes)

Output final: `best_distilbert_predictions.joblib`  
Columnas: `PetID | AdoptionSpeed | pred` (vector de 5 probabilidades)


## Paso 1: Instalar librerías


In [1]:
!pip install -q numpy==1.26.4
!pip install -q datasets==2.20.0 transformers==4.40.1
!pip install -q optuna
print('Librerías instaladas')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 40.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompat

In [ ]:
# Esta celda reinicia el kernel de Colab intencionalmente.
# Se hace ya que cuando se instalan paquetes con pip en una sesión de Python que ya está corriendo, los cambios no se aplican automáticamente a la sesión actual.
# Por lo tanto, para que las versiones nuevas de transformers, datasets, etc. queden activas, se debe interrumpir el proceso y reiniciar la sesión de colab.

import os
os.kill(os.getpid(), 9)


## Paso 2: Imports y configuración


In [1]:
# Data processing
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
import copy
import time
import datetime
from sklearn.metrics import cohen_kappa_score
from datasets import Dataset, DatasetDict
import optuna
from joblib import dump

# Modeling
import torch
from torch.utils.data import DataLoader
from transformers import (DistilBertTokenizerFast, DataCollatorWithPadding,
                          AutoModelForSequenceClassification, AdamW, get_scheduler)
from tqdm.auto import tqdm

print('CUDA disponible:', torch.cuda.is_available())


CUDA disponible: True


In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR = '/content/drive/MyDrive/PETFINDER_GRUPO8_R10/'
print(f'Carpeta base: {BASE_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Carpeta base: /content/drive/MyDrive/PETFINDER_GRUPO8_R10/


**Bajar el tokenizador**

El tokenizador convierte el texto de las descripciones en números que el modelo puede entender. Se descarga la versión preentrenada de DistilBERT desde HuggingFace, que sabe cómo dividir palabras en tokens y asignarles un ID numérico.


In [5]:
from transformers import DistilBertTokenizerFast
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

**Paths y parámetros**


In [6]:
# Paths
# BASE_DIR = './'   Fue definida más arriba, no redefinir
PATH_TO_TRAIN          = os.path.join(BASE_DIR, 'input/petfinder-adoption-prediction/train/train.csv')
PATH_TO_TEMP_FILES     = os.path.join(BASE_DIR, 'Texto/work/optuna_temp_artifacts')
PATH_TO_OPTUNA_ARTIFACTS = os.path.join(BASE_DIR, 'Texto/work/optuna_artifacts')
WORK_DIR               = os.path.join(BASE_DIR, 'Texto/work')

os.makedirs(PATH_TO_TEMP_FILES, exist_ok=True)
os.makedirs(PATH_TO_OPTUNA_ARTIFACTS, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)

# Parámetros — deben coincidir con el split del notebook tabular
SEED         = 42
TEST_SIZE    = 0.2
BATCH_SIZE   = 32                       # Se utiliza 32 por restricciones de GPU
MODEL_NAME   = '06_distilbert'
MODEL_VERSION = '1.0'


**Armado de los Datasets**

Se utiliza el mismo SEED=42 y TEST_SIZE=0.2 que el notebook tabular para garantizar que el split de validación sea idéntico y los PetID coincidan en el ensemble.


In [7]:
# Cargar los datos y split estratificado — igual que tabular (SEED=42, TEST_SIZE=0.2)

# Se carga el CSV y se filtran las mascotas sin descripción (solo 13 casos).
# Se crea la columna labels como copia de AdoptionSpeed, que es lo que el modelo va a predecir.
# Luego se hace el split 80/20 estratificado, garantizando que la proporción de cada clase se mantenga igual en train y test.
# El reset_index ordena los índices tras el split, lo que evita problemas al alinear predicciones con PetIDs más adelante.

# Cargar sin filtrar los nulos. Para que los dataset de train y test sean iguales al tabular.
df = pd.read_csv(PATH_TO_TRAIN)
df['labels'] = df['AdoptionSpeed']

# Split sobre el dataset completo — igual que tabular
train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=SEED, stratify=df.AdoptionSpeed)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print('ANTES de filtrar nulos:')
print(list(test_df['PetID'].head(10)))

# Filtrar descripción vacía DESPUÉS del split
train_df = train_df[train_df['Description'].notnull()].reset_index(drop=True)
test_df  = test_df[test_df['Description'].notnull()].reset_index(drop=True)

print('\nDESPUÉS de filtrar nulos:')
print(list(test_df['PetID'].head(10)))


print(f'Train: {len(train_df):,} | Test: {len(test_df):,}')
print('Distribución AdoptionSpeed (test):')
print(test_df['AdoptionSpeed'].value_counts().sort_index())


ANTES de filtrar nulos:
['8f20e24ef', '2d72ef0c4', '44cd12263', '210c4a637', '21493e6ea', '65ba8b6a0', 'bef935312', '157e3e074', '0ac71c2c1', '128293a4d']

DESPUÉS de filtrar nulos:
['8f20e24ef', '2d72ef0c4', '44cd12263', '210c4a637', '21493e6ea', '65ba8b6a0', 'bef935312', '157e3e074', '0ac71c2c1', '128293a4d']
Train: 11,984 | Test: 2,996
Distribución AdoptionSpeed (test):
AdoptionSpeed
0     82
1    618
2    805
3    652
4    839
Name: count, dtype: int64


In [8]:
# Antes de tokenizar, se guardan los PetID y AdoptionSpeed del set de test en listas separadas (son los identificadores que después se necesitan para armar el archivo del ensemble).

test_sample_ids          = list(test_df['PetID'])
test_adoption_speed      = list(test_df['AdoptionSpeed'])

# Los DataFrames se convierten al formato Dataset de HuggingFace, que es el que espera DistilBERT.
# El class_encode_column('labels') convierte los números 0-4 a un tipo categórico formal.
# cols_to_remove lista todas las columnas que no son labels para eliminarlas antes de tokenizar, ya que el modelo solo necesita el texto y la etiqueta.

train_dataset = Dataset.from_pandas(train_df)
test_dataset  = Dataset.from_pandas(test_df)
dataset = DatasetDict({'train': train_dataset, 'val': test_dataset})
dataset = dataset.class_encode_column('labels')

cols_to_remove = [col for col in dataset['train'].column_names if col != 'labels']
print('Columnas a remover antes de tokenizar:', cols_to_remove)


Stringifying the column:   0%|          | 0/11984 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/11984 [00:00<?, ? examples/s]

Stringifying the column:   0%|          | 0/2996 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/2996 [00:00<?, ? examples/s]

Columnas a remover antes de tokenizar: ['Type', 'Name', 'Age', 'Breed1', 'Breed2', 'Gender', 'Color1', 'Color2', 'Color3', 'MaturitySize', 'FurLength', 'Vaccinated', 'Dewormed', 'Sterilized', 'Health', 'Quantity', 'Fee', 'State', 'RescuerID', 'VideoAmt', 'Description', 'PetID', 'PhotoAmt', 'AdoptionSpeed']


**Tokenización**

Convierte las descripciones de texto en datos numéricos que DistilBERT puede procesar.

padding=True rellena las descripciones cortas para que todas tengan la misma longitud dentro del batch.

truncation=True corta las que superan max_length=512 tokens.

Se aplica sobre todo el dataset en paralelo (num_proc=2) y se eliminan todas las columnas que no son necesarias para el modelo.

Al final quedan solo tres columnas: input_ids (los tokens), attention_mask (indica qué tokens son reales y cuáles son padding) y labels.

In [9]:
def tokenize(batch):
    from transformers import DistilBertTokenizerFast
    tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
    return tokenizer(batch['Description'], padding=True, truncation=True, max_length=512)

dataset_enc = dataset.map(tokenize, batched=True, remove_columns=cols_to_remove, num_proc=2)
dataset_enc.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
print(dataset_enc['train'].column_names)


Map (num_proc=2):   0%|          | 0/11984 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Map (num_proc=2):   0%|          | 0/2996 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


['labels', 'input_ids', 'attention_mask']


**DataLoaders**

Envuelven el dataset en un objeto que entrega los datos en batches durante el entrenamiento.

El DataCollatorWithPadding se encarga de ajustar dinámicamente el padding batch a batch, siendo más eficiente que usar un padding fijo global.

El loader de train tiene shuffle=True para aleatorizar el orden en cada época y evitar que el modelo aprenda el orden de los datos; el de validación no lo necesita.


In [10]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_dataloader = DataLoader(
    dataset_enc['train'], shuffle=True, batch_size=BATCH_SIZE, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    dataset_enc['val'], batch_size=BATCH_SIZE, collate_fn=data_collator
)


**Cargamos el modelo**

Descarga DistilBERT preentrenado desde HuggingFace y reemplaza su cabeza de clasificación final para que prediga 5 clases (las categorías de AdoptionSpeed) en lugar de las que venía configurado originalmente.

Luego detecta si hay GPU disponible y mueve el modelo a ese dispositivo con model.to(device) — sin esto el entrenamiento correría en CPU y tardaría mucho más tiempo.


In [11]:
num_labels = dataset['train'].features['labels'].num_classes
print(f'Number of labels: {num_labels}')

model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=num_labels)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
model.to(device)


Number of labels: 5


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Device: cuda


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
 

## Paso 3: Función de entrenamiento

Entrena el modelo por N épocas y evalúa con QWK en cada una.  
Guarda los mejores pesos y, cuando se usa con Optuna, registra el mejor trial.

`train_val` devuelve también `best_output_scores` — el vector de probabilidades (5 clases) del mejor epoch, necesario para armar el output del ensemble.


In [12]:
def train_val(model, dataloaders, datasets, device, num_epochs=4, lr=0.001, trial=None):

    since = time.time()
    optimizer = AdamW(model.parameters(), lr=lr)

    num_training_steps = num_epochs * len(train_dataloader)
    lr_scheduler = get_scheduler('linear', optimizer=optimizer,
                                 num_warmup_steps=0,
                                 num_training_steps=num_training_steps)

    best_model_wts    = copy.deepcopy(model.state_dict())
    best_kappa        = -999.0
    best_output_scores = []

    try:
        previous_best = study.best_value
    except:
        previous_best = -999.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()

            running_loss    = 0.0
            running_corrects = 0
            kappa_true, kappa_pred = [], []
            output_scores   = []

            for batch in tqdm(dataloaders[phase], desc=phase):
                batch  = {k: v.to(device) for k, v in batch.items()}
                labels = batch['labels']
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs    = model(**batch)
                    loss       = outputs.loss
                    probs      = torch.nn.functional.softmax(outputs.logits, dim=-1)
                    preds_lbl  = torch.argmax(probs, dim=-1)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                        lr_scheduler.step()
                    else:
                        kappa_true.extend(labels.cpu().numpy().tolist())
                        kappa_pred.extend(preds_lbl.cpu().numpy().tolist())
                        probs_np = probs.detach().cpu().numpy()
                        output_scores.extend([probs_np[i, :] for i in range(probs_np.shape[0])])

                running_loss     += loss.item() * labels.size(0)
                running_corrects += torch.sum(preds_lbl == labels.data)

            epoch_loss = running_loss / len(datasets[phase])
            epoch_acc  = running_corrects.double() / len(datasets[phase])
            kappa_score = (cohen_kappa_score(kappa_true, kappa_pred, weights='quadratic')
                           if phase == 'val' else float('nan'))

            print(f'{phase.title()} | Loss: {epoch_loss:.4f} | '
                  f'Acc: {epoch_acc*100:.2f}% | QWK: {kappa_score}')

            if phase == 'val' and kappa_score > best_kappa:
                best_kappa        = kappa_score
                best_model_wts    = copy.deepcopy(model.state_dict())
                best_output_scores = output_scores

                if trial is not None:
                    trial.report(best_kappa, epoch)
                    if trial.should_prune():
                        raise optuna.exceptions.TrialPruned()

    time_elapsed = time.time() - since
    print(f'\nTiempo: {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Mejor QWK: {best_kappa:.4f}')

    model.load_state_dict(best_model_wts)
    return model, best_kappa, best_output_scores  #  devuelve 3 valores


## Paso 4: Entrenamiento base (sin Optuna)

Se entrena con hiperparámetros por defecto para tener un kappa de referencia.


In [13]:
best_model, best_kappa, best_scores_base = train_val(
    model,
    dataloaders={'train': train_dataloader, 'val': eval_dataloader},
    datasets=dataset_enc,
    device=device,
    lr=5e-5,
    num_epochs=3
)
print(f'\nKappa base: {best_kappa:.4f}')


/usr/local/lib/python3.12/dist-packages/transformers/optimization.py:521: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 0/2
----------


train:   0%|          | 0/375 [00:00<?, ?it/s]

Train | Loss: 1.4444 | Acc: 31.54% | QWK: nan


val:   0%|          | 0/94 [00:00<?, ?it/s]

...

Val | Loss: 1.4081 | Acc: 37.88% | QWK: 0.16793858521885574
Epoch 2/2
----------


train:   0%|          | 0/375 [00:00<?, ?it/s]

Train | Loss: 1.1784 | Acc: 49.67% | QWK: nan


val:   0%|          | 0/94 [00:00<?, ?it/s]

Val | Loss: 1.4271 | Acc: 36.85% | QWK: 0.24106635076175587

Tiempo: 29m 38s
Mejor QWK: 0.2411

Kappa base: 0.2411


**Guardar el modelo base**


In [14]:
from google.colab import files

run_id     = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
file_name  = f'{MODEL_NAME}_{MODEL_VERSION}_{run_id}.pth'
model_path = os.path.join(PATH_TO_TEMP_FILES, file_name)
torch.save(best_model, model_path)
print(f'Modelo guardado en {model_path}')

#files.download(model_path)
#print('Descarga iniciada')


Modelo guardado en /content/drive/MyDrive/PETFINDER_GRUPO8_R10/Texto/work/optuna_temp_artifacts/06_distilbert_1.0_20260530_121041.pth


## Paso 5: Optimización de hiperparámetros con Optuna

Optuna prueba distintas combinaciones de `lr` y `epochs` para maximizar el QWK.

Cada trial entrena el modelo completo. Con `n_trials=5` y épocas 3-4 es manejable en Colab T4.


In [ ]:
def optuna_train(trial):
    epochs = trial.suggest_int('epochs', 3, 4)
    lr     = trial.suggest_float('lr', 1e-5, 1e-4, log=True)

    model_trial = AutoModelForSequenceClassification.from_pretrained(
        'distilbert-base-uncased', num_labels=num_labels
    )
    model_trial.to(device)

    _, best_score, _ = train_val(
        model_trial,
        dataloaders={'train': train_dataloader, 'val': eval_dataloader},
        datasets=dataset_enc,
        device=device,
        num_epochs=epochs,
        lr=lr,
        trial=trial
    )
    return best_score


In [ ]:
# Descomentar para borrar el estudio y empezar desde cero
# db_path = '/content/drive/MyDrive/optuna_study.db'
# if os.path.exists(db_path):
#     os.remove(db_path)
#     print('Base de datos eliminada')


In [ ]:
study = optuna.create_study(
    direction    = 'maximize',
    study_name   = f'{MODEL_NAME}_{MODEL_VERSION}',
    storage      = 'sqlite:////content/drive/MyDrive/PETFINDER_GRUPO8_R10/Texto/work/optuna_study.db',
    load_if_exists = True
)
study.optimize(optuna_train, n_trials=5)

print('\nOptimización terminada')
print(f'Mejor kappa : {study.best_value:.4f}')
print(f'Mejores hiperparámetros: {study.best_params}')


## Paso 6: Reentrenar con los mejores hiperparámetros

Entrenamos el modelo final con los parámetros encontrados por Optuna.

En caso de que el modelo base de un mejor kappa, se saltea este paso.


In [15]:
try:
    if best_kappa > study.best_value:
        print(f'Optuna no mejoró el kappa base ({best_kappa:.4f} vs {study.best_value:.4f})')
        print('Saltando reentrenamiento — se usará el modelo base en el ensemble.')
    else:
        best_lr     = study.best_params['lr']
        best_epochs = study.best_params['epochs']
        print(f'Mejores parámetros: lr={best_lr:.6f}, epochs={best_epochs}')

        final_model = AutoModelForSequenceClassification.from_pretrained(
            'distilbert-base-uncased', num_labels=num_labels
        )
        final_model.to(device)

        final_model, final_kappa, final_scores = train_val(
            final_model,
            dataloaders={'train': train_dataloader, 'val': eval_dataloader},
            datasets=dataset_enc,
            device=device,
            lr=best_lr,
            num_epochs=best_epochs
        )
        print(f'\nKappa modelo final: {final_kappa:.4f}')
except NameError:
    print('Optuna no fue ejecutado — se usará el modelo base en el ensemble.')

Optuna no fue ejecutado — se usará el modelo base en el ensemble.


**Guardar el modelo final**


In [16]:
try:
    run_id     = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    file_name  = f'{MODEL_NAME}_{MODEL_VERSION}_final_{run_id}.pth'
    model_path = os.path.join(PATH_TO_TEMP_FILES, file_name)
    torch.save(final_model, model_path)
    print(f'Modelo guardado en {model_path}')

    #files.download(model_path)
    #print('Descarga iniciada')
except NameError:
    print('No hay modelo final — Optuna no mejoró el kappa base, no hay nada que guardar.')


No hay modelo final — Optuna no mejoró el kappa base, no hay nada que guardar.


## Paso 7: Guardar predicciones para el Ensemble

Genera `best_distilbert_predictions.joblib` con el mismo formato que `best_lgb_predictions.joblib`:

| Columna | Descripción |
|---|---|
| `PetID` | Identificador único de la mascota |
| `AdoptionSpeed` | Etiqueta real (0–4) |
| `pred` | Vector de 5 probabilidades (una por clase) |

Este archivo es la entrada directa del script de ensemble.


In [17]:
# Seleccionamos los mejores scores comparando kappa base vs final
# - Si Optuna se ejecutó: usamos final_scores o best_scores_base según cuál dé mejor kappa.
# - Si se saltó Optuna:   usamos best_scores_base

try:
    if final_kappa >= best_kappa:
        scores_para_ensemble = final_scores
        kappa_para_ensemble  = final_kappa
        print(f'Usando modelo post-Optuna (kappa: {final_kappa:.4f})')
    else:
        scores_para_ensemble = best_scores_base
        kappa_para_ensemble  = best_kappa
        print(f'Usando modelo base (kappa: {best_kappa:.4f}) — Optuna no mejoró')
except NameError:
    scores_para_ensemble = best_scores_base
    kappa_para_ensemble  = best_kappa
    print(f'Usando modelo base (kappa: {best_kappa:.4f}) — Optuna no fue ejecutado')

# Construimos el DataFrame — mismo formato que best_lgb_predictions.joblib
best_predicted_df = pd.DataFrame({
    'PetID':         test_sample_ids,
    'AdoptionSpeed': test_adoption_speed,
    'pred':          scores_para_ensemble,
})

# Verificaciones de integridad
assert len(best_predicted_df) == len(test_sample_ids), "Cantidad de filas no coincide"
assert best_predicted_df['PetID'].notna().all(),        "Hay PetID faltantes"
assert best_predicted_df['AdoptionSpeed'].notna().all(),"Hay AdoptionSpeed faltantes"
assert len(best_predicted_df['pred'][0]) == 5,          "El vector pred debe tener 5 clases"

print(f'\nDataFrame de ensemble listo')
print(f'   Filas: {len(best_predicted_df)} | PetIDs únicos: {best_predicted_df["PetID"].nunique()}')
print(f'   QWK del modelo: {kappa_para_ensemble:.4f}')
print('\nPrimeras filas:')
print(best_predicted_df[['PetID', 'AdoptionSpeed', 'pred']].head(5).to_string())


Usando modelo base (kappa: 0.2411) — Optuna no fue ejecutado

DataFrame de ensemble listo
   Filas: 2996 | PetIDs únicos: 2996
   QWK del modelo: 0.2411

Primeras filas:
       PetID  AdoptionSpeed                                                             pred
0  8f20e24ef              4  [0.013628965, 0.033669807, 0.39075443, 0.53999496, 0.021951843]
1  2d72ef0c4              4   [0.009532803, 0.043138288, 0.24359576, 0.27844647, 0.42528665]
2  44cd12263              4   [0.008052966, 0.024204707, 0.09409033, 0.13844553, 0.73520654]
3  210c4a637              2     [0.021581877, 0.3725048, 0.28616464, 0.24109685, 0.07865176]
4  21493e6ea              4      [0.04804152, 0.36626095, 0.2471839, 0.17540812, 0.16310553]


In [18]:
# Guardamos en work/
output_path = os.path.join(WORK_DIR, 'best_distilbert_predictions.joblib')
dump(best_predicted_df, output_path)

print(f'Archivo guardado en: {output_path}')
print(f'   Nombre esperado por el ensemble: best_distilbert_predictions.joblib')

# Descargarlo a la computadora (opcional)
#files.download(output_path)
#print('Descarga iniciada')


Archivo guardado en: /content/drive/MyDrive/PETFINDER_GRUPO8_R10/Texto/work/best_distilbert_predictions.joblib
   Nombre esperado por el ensemble: best_distilbert_predictions.joblib
